# PART 05 · 1장 작업형 1 핵심 패턴

교재 402~427쪽 본문 소스 코드입니다. 노트북과 같은 위치에 `data` 폴더를 두고 위에서부터 차례대로 실행하세요.

#### 1. 전체의 분석 흐름

#### 2. 공통 라이브러리와 데이터 불러오기

이 장에서 공통으로 사용하는 라이브러리와 네 개의 데이터를 한 번에 불러온다.

In [1]:
import pandas as pd
import numpy as np
filter_sort = pd.read_csv(
    'data/part02_type1/type1_filter_sort.csv'
)
missing_outlier = pd.read_csv(
    'data/part02_type1/type1_missing_outlier.csv'
)
sales = pd.read_csv(
    'data/part02_type1/type1_derived_group.csv'
)
reservations = pd.read_csv(
    'data/part02_type1/type1_date_string.csv'
)
print(filter_sort.shape)
print(missing_outlier.shape)
print(sales.shape)
print(reservations.shape)

(300, 8)
(350, 8)
(500, 9)
(400, 7)


## 01 조건 필터링 패턴

### 1 조건식 만들기

#### 1. 조건식과 불리언 인덱싱

#### 2. 단일 조건과 다중 조건

### 2 조건 확장과 계산

#### 1. 목록·범위·제외 조건

#### 2. 조건 적용 후 계산

조건에 맞는 행을 골랐다면 문제에서 요구한 계산을 마지막에 적용한다. 행의 개수는 len( ) 또는 조건식의 sum( ), 특정 컬럼의 합계·평균은 sum( )과 mean( )으로 구한다.

In [2]:
cond1 = filter_sort['age'] >= 40
print('age_40_or_more: ', len(filter_sort[cond1]))
cond2 = (
    (filter_sort['region'] == 'central')
    & (filter_sort['score'] >= 75)
)
print('central_score_75_count: ', cond2.sum())
print(
    'central_score_75_spend_mean: ',
    round(filter_sort.loc[cond2, 'total_spend'].mean(), 2)
)
cond3 = filter_sort['region'].isin(['north', 'east'])
print(
    'north_east_purchase_sum: ',
    filter_sort.loc[cond3, 'purchase_count'].sum()
)
cond4 = (
    filter_sort['score'].between(60, 80)
    & (filter_sort['segment'] != 'B')
)
print('score_60_80_not_B: ', cond4.sum())

age_40_or_more:  119
central_score_75_count:  23
central_score_75_spend_mean:  183213.04
north_east_purchase_sum:  975
score_60_80_not_B:  104


## 02 정렬 후 추출 패턴

### 1 기본 정렬과 추출

#### 1. 오름차순과 내림차순 정렬

#### 2. 순위를 위치로 바꾸기

### 2 정렬 응용

#### 1. 여러 컬럼을 기준으로 정렬하기

정렬 기준이 두 개 이상이면 sort_values( )에 컬럼명 리스트를 넣는다. 앞의 컬럼이 우선 기준이 되고, 값이 같을 때 다음 컬럼으로 순서를 정한다. ascending 리스트의 순서도 컬럼 리스트와 맞춰야 한다.

In [3]:
multi_sorted = filter_sort.sort_values(
    ['purchase_count', 'score'],
    ascending=[False, False]
)
print(
    multi_sorted[
        ['customer_id', 'purchase_count', 'score']
    ].head()
)

    customer_id  purchase_count  score
242      FS0243              19   41.9
6        FS0007              17   61.9
203      FS0204              16   64.6
54       FS0055              15   74.7
237      FS0238              15   73.2


#### 2. 조건 필터링 후 정렬·추출하기

조건 필터링, 정렬, 추출을 한 흐름으로 이어서 실행해본다.

In [4]:
spend_sorted = filter_sort.sort_values(
    'total_spend', ascending=False
)
print(
    'top: ',
    spend_sorted.iloc[0]['customer_id'],
    int(spend_sorted.iloc[0]['total_spend'])
)
print(
    'third: ',
    spend_sorted.iloc[2]['customer_id'],
    int(spend_sorted.iloc[2]['total_spend'])
)
central_sorted = filter_sort[
    filter_sort['region'] == 'central'
].sort_values('total_spend', ascending=False)
print(
    'central_second: ',
    central_sorted.iloc[1]['customer_id'],
    int(central_sorted.iloc[1]['total_spend'])
)

top:  FS0057 501400
third:  FS0191 415400
central_second:  FS0002 390000


## 03 결측치 처리 패턴

### 1 결측치 확인과 처리

#### 1. 결측치 확인하기

두 컬럼의 결측치 개수부터 확인해본다.

In [5]:
print(
    missing_outlier[['rating', 'views']]
    .isna()
    .sum()
)

rating    24
views     18
dtype: int64


#### 2. 결측치 대체하기

#### 3. 결측치가 있는 행 제거하기

dropna( )를 아무 조건 없이 사용하면 다른 컬럼의 결측치까지 기준이 될 수 있다. 문제에서 지정한 컬럼만 기준으로 제거할 때는 subset을 사용한다.

In [6]:
drop_df = missing_outlier.dropna(
    subset=['rating', 'views']
)
print(drop_df.shape)
print('removed: ', len(missing_outlier)-len(drop_df))

(308, 8)
removed:  42


### 2 처리 후 계산과 응용

#### 1. 결측치 처리 후 계산하기

결측치를 처리한 뒤 조건 계산까지 이어지는 패턴을 실행해본다.

In [7]:
work = missing_outlier.copy()
rating_median = work['rating'].median()
views_min = work['views'].min()
work['rating'] = work['rating'].fillna(rating_median)
work['views'] = work['views'].fillna(views_min)
cond = work['rating'] >= 4.5
print('rating_median: ', rating_median)
print('views_min: ', int(views_min))
print('rating_4_5_count: ', cond.sum())
print(
    'rating_4_5_mean_views: ',
    int(work.loc[cond, 'views'].mean())
)

rating_median:  3.9
views_min:  14
rating_4_5_count:  76
rating_4_5_mean_views:  14349


#### 2. 그룹별 대푯값으로 결측치 채우기

transform( )은 그룹별 계산값을 원본과 같은 길이와 인덱스로 반환하므로 각 행이 속한 그룹의 중앙값을 fillna( )에 바로 사용할 수 있다. 한 그룹의 대상 컬럼이 모두 결측이면 그룹 중앙값도 결측이므로, 전체 중앙값으로 한 번 더 보완한다. 컬럼 전체가 결측인 경우에는 전체 중앙값도 계산할 수 없으므로 문제에서 정한 상수나 별도 대체 기준이 필요하다.

컬럼 전체를 아래쪽 정수로 내릴 때는 np.floor( )를 적용한 뒤 astype(int)로 바꾼다. int( )는 0을 향해 소수점 이하를 버리므로 음수에서는 np.floor( )와 결과가 다를 수 있다.

In [8]:
filled_views = missing_outlier.copy()
group_median = filled_views.groupby(
    'item_group'
)['views'].transform('median')
overall_median = filled_views['views'].median()
fill_values = group_median.fillna(overall_median)
filled_views['views'] = (
    filled_views['views'].fillna(fill_values)
)
filled_views['views_floor'] = (
    np.floor(filled_views['views']).astype(int)
)
print('missing: ', filled_views['views'].isna().sum())
print(
    filled_views.loc[
        missing_outlier['views'].isna(),
        ['row_id', 'item_group', 'views', 'views_floor']
    ].head(3).to_string(index=False)
)

missing:  0
row_id item_group   views  views_floor
MO0008      alpha 15652.5        15652
MO0073       beta 13481.0        13481
MO0131      alpha 15652.5        15652


## 04 이상치 처리 패턴

### 1 IQR 기준과 이상치 찾기

#### 1. IQR 기준 계산하기

#### 2. 이상치 찾기

하한보다 작거나 상한보다 큰 값이 이상치다. 두 조건 중 하나만 만족하면 되므로 |를 사용한다. 정상 범위만 남기려면 이상치 조건 앞에 ~를 붙인다.

In [9]:
q1 = missing_outlier['price'].quantile(0.25)
q3 = missing_outlier['price'].quantile(0.75)
iqr = q3-q1
lower = q1-1.5*iqr
upper = q3+1.5*iqr
outlier_cond = (
    (missing_outlier['price'] < lower)
    | (missing_outlier['price'] > upper)
)
outliers = missing_outlier[outlier_cond]
print(outliers[['row_id', 'item_group', 'price']])

     row_id item_group     price
3    MO0004       beta  180000.0
101  MO0102      gamma  220000.0
207  MO0208       beta  195000.0


### 2 제거와 대체

#### 1. 이상치 제거하기

이상치를 제거하려면 이상치 조건을 반대로 적용한다. 이상치 세 행을 제외하면 347행이 남고, price 평균은 36778.67이다.

In [10]:
normal = missing_outlier[~outlier_cond]
print('normal_rows: ', len(normal))
print('normal_price_mean: ', round(normal['price'].mean(), 2))

normal_rows:  347
normal_price_mean:  36778.67


#### 2. 이상치를 경곗값으로 대체하기

이상치를 제거하는 대신 경곗값으로 대체하는 패턴을 실행해본다.

In [11]:
capped = missing_outlier.copy()
capped.loc[capped['price'] < lower, 'price'] = lower
capped.loc[capped['price'] > upper, 'price'] = upper
print('outlier_count: ', outlier_cond.sum())
print('outlier_price_sum: ', int(outliers['price'].sum()))
print('capped_price_mean: ', round(capped['price'].mean(), 2))
print('capped_price_sum: ', int(capped['price'].sum()))

outlier_count:  3
outlier_price_sum:  595000
capped_price_mean:  37136.39
capped_price_sum:  12997737


## 05 파생변수 생성 패턴

### 1 파생변수 만들기

#### 1. 계산식으로 파생변수 만들기

#### 2. 조건에 따라 값 부여하기

비교 조건을 새 컬럼에 저장하면 True/False 파생변수가 만들어진다. 구간별 등급이 필요하면 pd.cut( )을 사용할 수 있다. 이때 구간 경계가 어느 쪽에 포함되는지 right 옵션을 확인한다.

In [12]:
sales['has_discount'] = sales['coupon_rate'] > 0
sales['high_order'] = sales['net_sales'] >= 500000
bins = [0, 100000, 300000, 500000, float('inf')]
labels = ['low', 'middle', 'high', 'premium']
sales['sales_level'] = pd.cut(
    sales['net_sales'],
    bins=bins,
    labels=labels,
    right=False
)

#### 3. 조건별로 다른 값 대입하기

pd.cut( )은 값의 크기로 구간을 나누지만, 범주값마다 다른 점수를 매겨야 할 때는 직접 대입한다. 새 컬럼을 기본값으로 만든 뒤 loc[조건, 컬럼]에 값을 대입하면 조건을 만족하는 행만 바꿀 수 있다.

이 예제에서는 예약 데이터를 사용해 방문과 취소의 순차이만 비교한다. 방문은 1, 취소는 -1, 예약과 노쇼는 분석 규칙상 0으로 둔다. 결과의 부호보다 차이의 크기가 중요하면 abs( )를 적용한다. abs( ).idxmax( )는 절댓값이 가장 큰 인덱스를 반환한다.

In [13]:
status_work = reservations.copy()
status_work['service_group'] = (
    status_work['service_code'].str.split('-').str[0]
)
status_work['status_value'] = 0
status_work.loc[
    status_work['status'] == '방문', 'status_value'
] = 1
status_work.loc[
    status_work['status'] == '취소', 'status_value'
] = -1
service_balance = status_work.groupby(
    'service_group'
)['status_value'].sum()
answer_group = service_balance.abs().idxmax()
answer_value = service_balance.loc[answer_group]
print(service_balance)
print(answer_group, answer_value)

service_group
BOOK    26
CARE    20
FIT     29
FOOD    16
HAIR    21
Name: status_value, dtype: int64
FIT 29


### 2 활용과 검산

#### 1. 파생변수를 이용해 계산하기

파생변수를 만들고 이를 이용한 조건 계산까지 이어서 실행해본다.

In [14]:
sales['gross_sales'] = sales['qty'] * sales['unit_price']
sales['discount_amount'] = (
    sales['gross_sales'] * sales['coupon_rate']
).round(0).astype(int)
sales['net_check'] = (
    sales['gross_sales'] - sales['discount_amount']
)
print('gross_sales_sum: ', int(sales['gross_sales'].sum()))
print(
    'discount_amount_sum: ',
    int(sales['discount_amount'].sum())
)
print('net_sales_sum: ', int(sales['net_sales'].sum()))
print('has_discount_count: ', int(sales['has_discount'].sum()))
print('high_order_count: ', int(sales['high_order'].sum()))
print(
    'high_order_mean: ',
    round(sales.loc[sales['high_order'], 'net_sales'].mean(), 2)
)

gross_sales_sum:  166901700
discount_amount_sum:  8912475
net_sales_sum:  157989225
has_discount_count:  252
high_order_count:  113
high_order_mean:  721086.02


#### 2. 파생변수 검산하기

원본 데이터에 이미 결과 컬럼이 있다면 새로 계산한 값과 비교하여 계산식을 검산할 수 있다. 이번 데이터에서는 net_check와 원본 net_sales가 다른 행이 0개다.

In [15]:
mismatch = sales['net_check'] != sales['net_sales']
print('mismatch_count: ', mismatch.sum())
print(sales['sales_level'].value_counts().sort_index())

mismatch_count:  0
sales_level
low        127
middle     153
high       107
premium    113
Name: count, dtype: int64


## 06 그룹별 집계 패턴

### 1 그룹 만들기와 집계

#### 1. 하나의 컬럼으로 그룹 만들기

하나의 컬럼을 기준으로 그룹별 합계와 평균을 계산해본다.

In [16]:
region_sales = sales.groupby('region')['net_sales'].sum()
category_mean = sales.groupby('category')['net_sales'].mean()
print(region_sales.sort_values(ascending=False))
print(category_mean.sort_values(ascending=False).round(2))

region
metro       57712475
river       37525570
coast       25858075
mountain    18971360
island      17921745
Name: net_sales, dtype: int64
category
stationery    384510.54
digital       333575.00
food          302197.61
home          289423.68
hobby         279113.29
Name: net_sales, dtype: float64


#### 2. 여러 통계량을 한 번에 계산하기

agg( )를 사용하면 주문 건수, 수량 합계, 순매출 합계와 평균을 한 번에 계산할 수 있다. 결과 컬럼의 이름을 직접 정하면 뒤에서 정렬하거나 조건을 적용하기 쉽다.

In [17]:
region_summary = sales.groupby('region').agg(
    order_count=('order_id', 'count'),
    qty_sum=('qty', 'sum'),
    net_sales_sum=('net_sales', 'sum'),
    net_sales_mean=('net_sales', 'mean')
).sort_values('net_sales_sum', ascending=False)
print(region_summary.round(2))

          order_count  qty_sum  net_sales_sum  net_sales_mean
region                                                       
metro             178     1360       57712475       324227.39
river             113      863       37525570       332084.69
coast              78      630       25858075       331513.78
mountain           71      499       18971360       267202.25
island             60      393       17921745       298695.75


#### 3. 여러 컬럼으로 그룹 만들기

그룹 기준이 두 개라면 groupby( )에 리스트를 넣는다. 결과는 다중 인덱스가 되므로 조건 필터링이나 일반적인 정렬을 이어갈 때는 reset_index( )로 컬럼으로 되돌리는 것이 편하다.

In [18]:
combo_summary = sales.groupby(
    ['region', 'category']
).agg(
    order_count=('order_id', 'count'),
    net_sales_sum=('net_sales', 'sum'),
    net_sales_mean=('net_sales', 'mean')
).reset_index()
combo_summary = combo_summary.sort_values(
    'net_sales_sum', ascending=False
).reset_index(drop=True)
print(combo_summary.head(6).round(2))

  region    category  order_count  net_sales_sum  net_sales_mean
0  metro        food           51       15671470       307283.73
1  metro  stationery           38       12216705       321492.24
2  metro     digital           30       11695925       389864.17
3  metro        home           30       11017410       367247.00
4  river  stationery           24       10745750       447739.58
5  coast  stationery           15        8197005       546467.00


### 2 집계 결과 활용

#### 1. 집계 결과를 표로 펼쳐 비교하기

지역별로 여러 카테고리의 값을 한 행에서 비교하려면 category를 열로 펼친 wide 형태가 편리하다. pivot_table( )은 원본 데이터에서 바로 표를 만들고, 다중 인덱스 집계 결과가 있다면 unstack( )으로 같은 형태를 만들 수 있다.비율을 계산할 때는 필요한 범주 열을 먼저 맞춘다. 분모가 0인 값은 NaN으로 바꾸고, 필요한 값이 없는 행을 제외한다. 유효한 비율이 남아 있는지 확인한 뒤 idxmax( )를 적용한다.

In [19]:
category_wide = sales.pivot_table(
    index='region',
    columns='category',
    values='net_sales',
    aggfunc='mean'
).reindex(columns=['food', 'home'])
food_base = category_wide['food'].replace(0, np.nan)
home_food_ratio = (
    category_wide['home'] / food_base
).dropna()
if home_food_ratio.empty:
    raise ValueError('비교할 수 있는 유효한 비율이 없습니다.')
answer_region = home_food_ratio.idxmax()
answer_ratio = home_food_ratio.loc[answer_region]
print(home_food_ratio.round(3))
print(answer_region, round(answer_ratio, 3))

region
coast       0.837
island      1.080
metro       1.195
mountain    1.330
river       0.591
dtype: float64
mountain 1.33


0 또는 결측 분모를 제외한 결과에서 mountain의 home 평균 순매출은 food 평균 순매출의 약 1.330배로 가장 크다.이미 두 기준으로 집계한 다중 인덱스 결과가 있다면 unstack( )으로 같은 형태를 만들 수 있다.

In [20]:
category_wide_2 = (
    sales.groupby(['region', 'category'])['net_sales']
    .mean()
    .unstack()
)

집계 기준이 열로 펼쳐진 wide 형태에서는 비교 방향을 먼저 정한다. 각 월에서 순매출 합계가 가장 큰 지역을 찾으려면 월별 열을 세로 방향으로 비교하므로 idxmax(axis=0)을 사용한다.

시작 시점과 끝 시점으로 기간당 복합 증가율을 계산할 때는 필요한 월 컬럼이 존재하는지 먼저 확인하고, 시작값이 0보다 큰 행만 계산한다. 이 예제에는 1월과 6월 데이터가 있으며 변화 구간 수는 두 월의 차이로 계산한다.

In [21]:
sales_wide = sales.copy()
sales_wide['order_month'] = pd.to_datetime(
    sales_wide['order_date']
).dt.month
monthly_sales = sales_wide.pivot_table(
    index='region',
    columns='order_month',
    values='net_sales',
    aggfunc='sum',
    fill_value=0
)
best_region_by_month = monthly_sales.idxmax(axis=0)
best_counts = best_region_by_month.value_counts()
most_frequent_region = best_counts.idxmax()
required_months = [1, 6]
has_months = set(required_months).issubset(
    monthly_sales.columns
)
if not has_months:
    raise ValueError('필요한 월 컬럼이 없습니다.')
start_month, end_month = required_months
period_count = end_month-start_month
start = monthly_sales[start_month]
end = monthly_sales[end_month]
valid = start.gt(0) & end.notna()
compound_growth = (
    (end[valid] / start[valid]) ** (1 / period_count) - 1
)*100
print('has_months: ', has_months)
print(best_region_by_month)
print(best_counts)
print('most_frequent_region: ', most_frequent_region)
print('metro_growth: ', round(compound_growth['metro'], 3))

has_months:  True
order_month
1    river
2    metro
3    metro
4    metro
5    metro
6    metro
dtype: object
metro    5
river    1
Name: count, dtype: int64
most_frequent_region:  metro
metro_growth:  7.712


#### 2. 그룹별 최고 원본 행 선택하기

그룹별 합계가 가장 큰 그룹을 찾는 것과 각 그룹에서 값이 가장 큰 원본 행을 찾는 것은 다르다. groupby( )[평가 컬럼].idxmax( )는 그룹마다 최댓값을 가진 원본 행의 인덱스를 반환한다.

이 인덱스를 원본 데이터프레임의 loc[]에 넣으면 식별자, 지역처럼 같은 행의 다른 값을 함께 확인할 수 있다. 그룹 안에 최댓값이 여러 개면 먼저 나타난 행의 인덱스가 반환된다.

In [22]:
sales_top = sales.copy()
sales_top['gross_sales'] = (
    sales_top['qty'] * sales_top['unit_price']
)
sales_top['discount_amount'] = (
    sales_top['gross_sales'] - sales_top['net_sales']
)
idx = sales_top.groupby(
    'category'
)['discount_amount'].idxmax()
top_discount_rows = sales_top.loc[
    idx,
    ['category', 'order_id', 'region', 'discount_amount']
].sort_values('category')
region_counts = top_discount_rows['region'].value_counts()
print(top_discount_rows.to_string(index=False))
print(region_counts)
print(region_counts.idxmax(), region_counts.max())

  category order_id   region  discount_amount
   digital  DG00375    river           119145
      food  DG00192    metro           142020
     hobby  DG00184 mountain           111780
      home  DG00147    metro           181720
stationery  DG00252    coast           140490
region
metro       2
river       1
mountain    1
coast       1
Name: count, dtype: int64
metro 2


#### 3. 집계 결과에서 답 찾기

집계 결과가 만들어졌다면 idxmax( ), max( ), sort_values( ), 조건 필터링을 이용해 최종 답을 찾는다. idxmax( )는 최댓값 자체가 아니라 최댓값을 가진 인덱스, 즉 그룹 이름을 반환한다.

In [23]:
best_region = region_summary['net_sales_sum'].idxmax()
category_summary = sales.groupby('category').agg(
    net_sales_mean=('net_sales', 'mean')
)
best_category = category_summary['net_sales_mean'].idxmax()
over_count = len(
    combo_summary[combo_summary['net_sales_sum'] >= 8000000]
)
print('best_region: ', best_region)
print('best_category_by_mean: ', best_category)
print('combo_over_8000000_count: ', over_count)

best_region:  metro
best_category_by_mean:  stationery
combo_over_8000000_count:  6


## 07 날짜 처리 패턴

### 1 날짜형 변환과 추출

#### 1. 문자열을 날짜형으로 변환하기

문자열 날짜 컬럼을 날짜형으로 변환하고 결과를 확인해본다.

In [24]:
reservations['reserved_dt'] = pd.to_datetime(
    reservations['reserved_at']
)
reservations['cancelled_dt'] = pd.to_datetime(
    reservations['cancelled_at']
)
print(reservations[['reserved_at', 'reserved_dt']].head())

        reserved_at         reserved_dt
0  2026-07-22 15:00 2026-07-22 15:00:00
1  2026-03-12 07:00 2026-03-12 07:00:00
2  2026-07-29 11:00 2026-07-29 11:00:00
3  2026-01-03 07:00 2026-01-03 07:00:00
4  2026-06-28 07:00 2026-06-28 07:00:00

#### 2. 날짜 요소 추출하기

### 2 날짜 조건과 계산

#### 1. 날짜 조건 적용하기

추출한 날짜 요소로 조건 계산을 실행해본다.

In [25]:
reservations['reserve_year'] = reservations['reserved_dt'].dt.year
reservations['reserve_month'] = reservations['reserved_dt'].dt.month
reservations['reserve_weekday'] = (
    reservations['reserved_dt'].dt.weekday
)
reservations['reserve_date'] = reservations['reserved_dt'].dt.date
may = reservations[
    (reservations['reserve_year'] == 2026)
    & (reservations['reserve_month'] == 5)
]
print('may_count: ', len(may))
print('may_visit_count: ', len(may[may['status'] == '방문']))
print(
    'weekend_count: ',
    (reservations['reserve_weekday'] >= 5).sum()
)

may_count:  49
may_visit_count:  22
weekend_count:  119


#### 2. 날짜별 빈도와 날짜 차이 계산하기

날짜별 빈도가 가장 높은 날짜와 날짜 차이 계산을 이어서 실행해본다.

In [26]:
date_counts = reservations['reserve_date'].value_counts()
top_date = date_counts.index[0]
top_date_count = date_counts.iloc[0]
reservations['cancel_days_before'] = (
    reservations['reserved_dt'].dt.normalize()
    - reservations['cancelled_dt'].dt.normalize()
).dt.days
print('top_date: ', top_date)
print('top_date_count: ', top_date_count)

top_date:  2026-08-06
top_date_count:  6


## 08 문자열 처리 패턴

### 1 문자열 조건과 추출

#### 1. 문자열 포함·시작·끝 조건

#### 2. 문자열 나누기와 일부 값 추출하기

service_code는 FOOD-50-Y처럼 서비스 그룹, 번호, 등급이 하이픈으로 연결되어 있다. split(‘-’)으로 나눈 뒤 str[0], str[1], str[2]로 각 조각을 꺼낸다.

In [27]:
parts = reservations['service_code'].str.split('-')
reservations['service_group'] = parts.str[0]
reservations['service_no_text'] = parts.str[1]
reservations['service_grade'] = parts.str[2]
reservations['phone_mid'] = (
    reservations['phone'].str.split('-').str[1]
)
print(
    reservations[
        ['service_code', 'service_group',
         'service_no_text', 'service_grade', 'phone_mid']
    ].head()
)

  service_code service_group service_no_text service_grade phone_mid
0    FOOD-50-Y          FOOD              50             Y      2076
1    FOOD-84-Y          FOOD              84             Y      6230
2    FOOD-66-Y          FOOD              66             Y      7660
3     FIT-48-X           FIT              48             X      5565
4    HAIR-62-Y          HAIR              62             Y      5449

str.split( ) 뒤에 str.len( )을 연결하면 각 문자열의 조각 수를 반복문 없이 계산할 수 있다.

In [28]:
codes = pd.Series(['BASIC', 'CARE-01', 'FOOD-20-Y'])
part_count = codes.str.split('-').str.len()
print(part_count.tolist())

[1, 2, 3]


### 2 변환과 활용

#### 1. 추출한 문자열의 자료형 변환

split( )으로 꺼낸 숫자는 문자형이다. ‘70 이상’처럼 크기를 비교하려면 astype(int)로 숫자형으로 바꿔야 한다. 문자 ‘100’과 ‘70’을 비교하면 숫자 비교와 다른 결과가 나올 수 있다.

In [29]:
reservations['service_no'] = (
    reservations['service_no_text'].astype(int)
)
print(reservations['service_no'].dtype)

int64


#### 2. 문자열 처리 후 필터링·집계하기

문자열 처리로 만든 컬럼에 필터링과 집계를 이어서 적용해본다.

In [30]:
food_y = (
    (reservations['service_group'] == 'FOOD')
    & (reservations['service_grade'] == 'Y')
)
hair_high = (
    (reservations['service_group'] == 'HAIR')
    & (reservations['service_no'] >= 70)
)
phone_mid_6 = reservations['phone_mid'].str.startswith('6')
contains_50 = reservations['service_code'].str.contains(
    '-50-', regex=False
)
visit_counts = reservations[
    reservations['status'] == '방문'
]['service_group'].value_counts()
print('food_y_count: ', food_y.sum())
print('hair_service_no_70_or_more: ', hair_high.sum())
print('phone_mid_startswith_6: ', phone_mid_6.sum())
print('service_code_contains_50: ', contains_50.sum())
print('top_visit_group: ', visit_counts.idxmax())
print('top_visit_count: ', visit_counts.max())

food_y_count:  29
hair_service_no_70_or_more:  29
phone_mid_startswith_6:  40
service_code_contains_50:  6
top_visit_group:  FIT
top_visit_count:  48
